# C-MAPSS — ICML/ICLR-oriented Heterogeneous Federated Multi-Task RUL & Early-Risk Learning

**Paper-oriented version — no security/attack section.**

The proposed method in this notebook is **HeteroFedMTL**: feature-heterogeneous federated multi-task prognostics with a sensor-set invariant encoder, adaptive multi-task weighting, task-aware server aggregation, and personalized residual heads. The two tasks are **RUL regression** and **early-failure-risk classification**.

The notebook intentionally uses the same model/training protocol as the other three dataset notebooks so results can be compared consistently.


### Dataset protocol
- **Clients:** FD001, FD002, FD003, FD004.
- **RUL:** standard capped C-MAPSS target (cap 125 cycles).
- **Auxiliary task:** failure-within-30-cycles classification. This is intentionally described as an *auxiliary early-risk task derived from RUL*, not as an independent fault-mode label.
- **Validation:** held-out training engines; **test:** official C-MAPSS test engines.


In [1]:
# Core imports and reproducibility
import os, glob, re, math, copy, random, warnings, time, json
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    average_precision_score, roc_auc_score,
    f1_score, precision_score, recall_score,
    brier_score_loss, log_loss, confusion_matrix
)
from scipy.stats import wilcoxon


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [2]:
# ============================================================
# Publication experiment configuration
# ============================================================
# Development mode keeps every experiment runnable quickly. For final paper
# numbers set PAPER_MODE=True and enable the experiment switches below.
PAPER_MODE = True

# Controlled feature heterogeneity used in the main experiment.
ENABLE_FEATURE_HETEROGENEITY = False
MIN_FEATURE_FRACTION = 0.65
FEATURE_HETEROGENEITY_MODE = "random"  # "random", "structured", or "none"
FAULT_FRACTION_BEARING = 0.40

ROUNDS = 50 if PAPER_MODE else 4
LOCAL_EPOCHS = 3 if PAPER_MODE else 1
BATCH_SIZE = 128
LR = 1e-3
D_MODEL = 64
HIDDEN = 96

# Proposed optimization: task-wise GroupDRO across clients + optional PCGrad.
DRO_ETA = 0.02
USE_PCGRAD = False
PARTICIPATION_RATE = 1.0

# Experiment switches. Final paper: enable them one section at a time.
RUN_MAIN_COMPARISON = True
RUN_MTL_STUDY = True
RUN_ABLATIONS = True
RUN_FEATURE_STRESS = True
RUN_NONIID_STRESS = True
RUN_PARTICIPATION_STRESS = True
RUN_UNSEEN_CLIENT = True
RUN_CALIBRATION = True
RUN_EFFICIENCY = True
RUN_MULTI_SEED = True
RUN_SENSITIVITY = True

N_SEEDS = 5
RESULT_DIR = Path("./heterofedmtl_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "PAPER_MODE": PAPER_MODE, "ROUNDS": ROUNDS, "LOCAL_EPOCHS": LOCAL_EPOCHS,
    "feature_mode": FEATURE_HETEROGENEITY_MODE, "min_feature_fraction": MIN_FEATURE_FRACTION,
    "DRO_ETA": DRO_ETA, "PCGrad": USE_PCGRAD
})


{'PAPER_MODE': True, 'ROUNDS': 50, 'LOCAL_EPOCHS': 3, 'feature_mode': 'random', 'min_feature_fraction': 0.65, 'DRO_ETA': 0.02, 'PCGrad': False}


In [3]:
# ============================================================
# Common data utilities
# ============================================================
def _pack_split(X, y, f, features, name=None):
    X = np.asarray(X, np.float32)
    y = np.asarray(y, np.float32).reshape(-1)
    f = np.asarray(f, np.float32).reshape(-1)
    assert len(X) == len(y) == len(f)
    return {"X": X, "y": y, "f": f, "features": list(features), "name": name}


def _chronological_split(X, y, f, features, train_frac=0.70, val_frac=0.15, name="client"):
    n = len(X)
    if n < 12:
        raise ValueError(f"{name}: too few windows ({n}) for train/val/test split")
    i1 = max(1, int(n * train_frac))
    i2 = max(i1 + 1, int(n * (train_frac + val_frac)))
    i2 = min(i2, n - 1)
    return {
        "name": name,
        "train": _pack_split(X[:i1], y[:i1], f[:i1], features, name),
        "val":   _pack_split(X[i1:i2], y[i1:i2], f[i1:i2], features, name),
        "test":  _pack_split(X[i2:], y[i2:], f[i2:], features, name),
    }


def robust_scale_client(c):
    """Leakage-safe robust scaling using this client's training windows only."""
    tr = c["train"]["X"]
    flat = tr.reshape(-1, tr.shape[-1])
    med = np.median(flat, axis=0)
    q1, q3 = np.percentile(flat, [25, 75], axis=0)
    scale = q3 - q1
    scale[scale < 1e-6] = 1.0
    out = {"name": c["name"]}
    for split in ("train", "val", "test"):
        s = c[split]
        X = np.clip((s["X"] - med) / scale, -8, 8).astype(np.float32)
        out[split] = _pack_split(X, s["y"], s["f"], s["features"], c["name"])
    return out


def apply_feature_heterogeneity(clients, min_fraction=0.65, seed=42, mode="random"):
    """Controlled client-specific sensor missingness without target leakage."""
    if mode == "none" or min_fraction >= 0.999:
        return clients
    out = []
    for cid, c in enumerate(clients):
        feats = c["train"]["features"]
        p = len(feats)
        if p <= 2:
            out.append(copy.deepcopy(c)); continue
        # Clients span min_fraction..1.0 so heterogeneity is not identical across clients.
        frac = min(1.0, min_fraction + (1.0-min_fraction) * ((cid % 5) / 4.0))
        keep_n = max(2, int(round(p * frac)))
        if mode == "structured":
            # Drop one contiguous feature block; this mimics a missing sensor family better
            # than independent random deletion while remaining dataset-agnostic.
            drop_n = p - keep_n
            start = (cid * max(1, p // max(len(clients), 1))) % p
            drop = {(start+j) % p for j in range(drop_n)}
            idx = np.asarray([j for j in range(p) if j not in drop], dtype=int)
        else:
            rng = np.random.default_rng(seed + 1009*cid)
            idx = np.sort(rng.choice(p, keep_n, replace=False))
        cc = {"name": c["name"]}
        for split in ("train", "val", "test"):
            s = c[split]
            cc[split] = _pack_split(s["X"][..., idx], s["y"], s["f"],
                                    [feats[i] for i in idx], c["name"])
        out.append(cc)
    return out


def build_feature_vocab(clients):
    return sorted({f for c in clients for f in c["train"]["features"]})


def assign_feature_ids(clients, vocab, allow_unknown=False):
    fmap = {f: i for i, f in enumerate(vocab)}
    unk = len(vocab)
    for c in clients:
        for split in ("train", "val", "test"):
            ids=[]
            for f in c[split]["features"]:
                if f in fmap: ids.append(fmap[f])
                elif allow_unknown: ids.append(unk)
                else: raise KeyError(f"Feature {f} absent from vocabulary")
            c[split]["feature_ids"] = np.asarray(ids, np.int64)
    return clients


def align_clients_to_union(clients, vocab):
    """Zero-fill missing sensors for conventional shared-input FL baselines."""
    pos = {f: i for i, f in enumerate(vocab)}
    aligned=[]
    for c in clients:
        cc={"name":c["name"]}
        for split in ("train","val","test"):
            s=c[split]
            X=np.zeros((*s["X"].shape[:-1],len(vocab)),np.float32)
            for j,f in enumerate(s["features"]): X[...,pos[f]]=s["X"][...,j]
            cc[split]=_pack_split(X,s["y"],s["f"],vocab,c["name"])
            cc[split]["feature_ids"]=np.arange(len(vocab),dtype=np.int64)
        aligned.append(cc)
    return aligned


def align_clients_to_intersection(clients):
    common=set(clients[0]["train"]["features"])
    for c in clients[1:]: common &= set(c["train"]["features"])
    common=sorted(common)
    if len(common)<2:
        raise ValueError("Feature intersection has fewer than 2 features")
    out=[]
    for c in clients:
        pos={f:i for i,f in enumerate(c["train"]["features"])}
        idx=[pos[f] for f in common]
        cc={"name":c["name"]}
        for split in ("train","val","test"):
            s=c[split]
            cc[split]=_pack_split(s["X"][...,idx],s["y"],s["f"],common,c["name"])
            cc[split]["feature_ids"]=np.arange(len(common),dtype=np.int64)
        out.append(cc)
    return out, common


def finalize_clients(raw_clients, min_fraction=MIN_FEATURE_FRACTION,
                     feature_mode=FEATURE_HETEROGENEITY_MODE, seed=SEED):
    cs=[robust_scale_client(copy.deepcopy(c)) for c in raw_clients]
    if ENABLE_FEATURE_HETEROGENEITY:
        cs=apply_feature_heterogeneity(cs,min_fraction,seed,feature_mode)
    vocab=build_feature_vocab(cs)
    assign_feature_ids(cs,vocab)
    rul_scale=max(max(float(np.max(c["train"]["y"])),1.0) for c in cs)
    print(f"Clients={len(cs)} | vocabulary={len(vocab)} | RUL scale={rul_scale:.2f}")
    for c in cs:
        print(f"  {c['name']:<14} train={len(c['train']['X']):6d} val={len(c['val']['X']):5d} "
              f"test={len(c['test']['X']):5d} features={len(c['train']['features']):2d} "
              f"risk%={100*c['train']['f'].mean():5.1f}")
    return cs,vocab,rul_scale


def induce_rul_distribution_shift(clients, severity=0.0, seed=42):
    """Controlled non-IID RUL-distribution shift with equal client sample counts.

    severity=0 returns an unchanged copy. At larger severity each client is resampled
    toward a different region of normalized RUL. Validation/test sets stay untouched.
    """
    out=copy.deepcopy(clients)
    if severity<=0: return out
    K=len(out)
    for cid,c in enumerate(out):
        s=c["train"]; y=s["y"].astype(float)
        yn=(y-y.min())/(max(y.max()-y.min(),1e-8))
        center=(cid+0.5)/K
        logits=-severity*((yn-center)**2)/0.08
        prob=np.exp(logits-logits.max()); prob/=prob.sum()
        rng=np.random.default_rng(seed+cid*97)
        idx=rng.choice(len(y),size=len(y),replace=True,p=prob)
        for key in ("X","y","f"): s[key]=s[key][idx]
    return out


In [4]:
# ============================================================
# C-MAPSS loader: FD001-FD004 as federated clients
# ============================================================
DATA_DIR = "/kaggle/input/datasets/palbha/cmapss-jet-engine-simulated-data"  # change for Colab/local
CLIENT_SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
WINDOW = 30
STRIDE = 1 if PAPER_MODE else 3
RUL_CAP = 125.0
FAULT_HORIZON = 30.0

CMAPSS_COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]


def _find_file(pattern):
    hits=glob.glob(os.path.join(DATA_DIR,"**",pattern),recursive=True)
    if not hits: raise FileNotFoundError(f"Cannot find {pattern} below {DATA_DIR}")
    return sorted(hits)[0]


def read_cmapss(subset):
    tr=pd.read_csv(_find_file(f"train_{subset}.txt"),sep=r"\s+",header=None,names=CMAPSS_COLS)
    te=pd.read_csv(_find_file(f"test_{subset}.txt"),sep=r"\s+",header=None,names=CMAPSS_COLS)
    offsets=pd.read_csv(_find_file(f"RUL_{subset}.txt"),sep=r"\s+",header=None).iloc[:,0].to_numpy(float)
    tr_max=tr.groupby("unit")["cycle"].transform("max")
    tr["RUL"]=np.minimum(tr_max-tr["cycle"],RUL_CAP)
    te_max=te.groupby("unit")["cycle"].max().sort_index()
    final_rul=dict(zip(te_max.index,offsets))
    te["RUL"]=[min((te_max.loc[u]+final_rul[u]-cy),RUL_CAP) for u,cy in zip(te.unit,te.cycle)]
    sensors=[f"s{i}" for i in range(1,22) if tr[f"s{i}"].std()>1e-8]
    return tr,te,sensors


def window_units(df,features,units,window=WINDOW,stride=STRIDE):
    X=[];y=[];f=[]
    for u in units:
        g=df[df.unit==u].sort_values("cycle")
        a=g[features].to_numpy(np.float32); r=g.RUL.to_numpy(np.float32)
        if len(a)<window:
            pad=window-len(a); a=np.vstack([np.repeat(a[:1],pad,0),a]); r=np.r_[np.repeat(r[:1],pad),r]
        for s in range(0,len(a)-window+1,stride):
            rr=float(r[s+window-1]); X.append(a[s:s+window]); y.append(rr); f.append(rr<=FAULT_HORIZON)
    return np.asarray(X,np.float32),np.asarray(y,np.float32),np.asarray(f,np.float32)


def build_cmapss_client(subset):
    tr,te,features=read_cmapss(subset)
    units=np.array(sorted(tr.unit.unique())); rng=np.random.default_rng(SEED+int(subset[-1])); rng.shuffle(units)
    nv=max(1,int(.20*len(units))); val_u=units[:nv]; train_u=units[nv:]
    Xtr,ytr,ftr=window_units(tr,features,train_u); Xv,yv,fv=window_units(tr,features,val_u)
    Xte,yte,fte=window_units(te,features,sorted(te.unit.unique()))
    return {"name":subset,
            "train":_pack_split(Xtr,ytr,ftr,features,subset),
            "val":_pack_split(Xv,yv,fv,features,subset),
            "test":_pack_split(Xte,yte,fte,features,subset)}

RAW_CLIENTS=[build_cmapss_client(s) for s in CLIENT_SUBSETS]
clients, FEATURE_VOCAB, RUL_SCALE = finalize_clients(copy.deepcopy(RAW_CLIENTS), MIN_FEATURE_FRACTION, FEATURE_HETEROGENEITY_MODE, SEED)


Clients=4 | vocabulary=21 | RUL scale=125.00
  FD001          train= 13937 val= 3794 test=10196 features=15 risk%= 17.8
  FD002          train= 37162 val= 9057 test=26511 features=21 risk%= 17.4
  FD003          train= 17708 val= 4112 test=13696 features=16 risk%= 14.0
  FD004          train= 43542 val=10486 test=34092 features=21 risk%= 14.2


## Proposed method — HeteroFedMTL-DRO

The method is formulated as **federated multi-task learning under joint feature and distribution heterogeneity**.
Each client may expose a different subset of sensors. A sensor-set encoder maps variable feature sets into a common latent space, followed by a shared temporal representation and two task heads: RUL regression and early failure-risk prediction. Small residual adapters remain private for personalization.

The server no longer uses heuristic quality weights. For each task, it maintains a client distribution and performs an **exponentiated-gradient GroupDRO update** so clients with high validation loss receive greater weight. RUL and risk heads therefore have different robust client weights. The shared representation uses the average of those two task distributions, which follows directly from an equal-weight sum of the two robust task objectives.

Locally, optional **PCGrad** projects conflicting RUL/risk gradients on the shared representation. This produces a testable hypothesis: robustness should improve as feature missingness, non-IID severity, client dropout, and unseen-client shift increase.


In [5]:
# ============================================================
# Proposed model: sensor-set invariant encoder + temporal backbone
# + shared task heads + private residual personalization
# ============================================================
class SensorSetEncoder(nn.Module):
    def __init__(self, n_features_vocab, d_model=64):
        super().__init__()
        self.sensor_emb=nn.Embedding(n_features_vocab,d_model)
        self.value_proj=nn.Sequential(nn.Linear(1,d_model),nn.GELU(),nn.Linear(d_model,d_model))
        self.gate=nn.Linear(d_model,1)
        self.norm=nn.LayerNorm(d_model)
    def forward(self,x,feature_ids):
        val=self.value_proj(x.unsqueeze(-1))
        sid=self.sensor_emb(feature_ids).view(1,1,-1,val.shape[-1])
        tok=self.norm(val+sid)
        a=torch.softmax(self.gate(tok).squeeze(-1),dim=2).unsqueeze(-1)
        return (a*tok).sum(dim=2)


class HeteroFedMTL(nn.Module):
    def __init__(self,vocab_size,rul_scale,d_model=D_MODEL,hidden=HIDDEN,personalization=True):
        super().__init__(); self.rul_scale=float(rul_scale); self.personalization=personalization
        self.sensor_encoder=SensorSetEncoder(vocab_size,d_model)
        self.temporal=nn.Sequential(nn.Conv1d(d_model,d_model,5,padding=2),nn.GELU(),
                                    nn.Conv1d(d_model,d_model,3,padding=1),nn.GELU())
        self.gru=nn.GRU(d_model,hidden,batch_first=True)
        self.trunk=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.LayerNorm(hidden))
        self.shared_rul_head=nn.Sequential(nn.Linear(hidden,48),nn.GELU(),nn.Linear(48,1))
        self.shared_risk_head=nn.Sequential(nn.Linear(hidden,48),nn.GELU(),nn.Linear(48,1))
        self.rul_adapter=nn.Linear(hidden,1); self.risk_adapter=nn.Linear(hidden,1)
        nn.init.zeros_(self.rul_adapter.weight); nn.init.zeros_(self.rul_adapter.bias)
        nn.init.zeros_(self.risk_adapter.weight); nn.init.zeros_(self.risk_adapter.bias)
        self.log_var_rul=nn.Parameter(torch.zeros(())); self.log_var_risk=nn.Parameter(torch.zeros(()))
    def latent(self,x,feature_ids):
        z=self.sensor_encoder(x,feature_ids)
        z=self.temporal(z.transpose(1,2)).transpose(1,2)
        z,_=self.gru(z)
        return self.trunk(z[:,-1])
    def forward(self,x,feature_ids):
        z=self.latent(x,feature_ids)
        r=self.shared_rul_head(z).squeeze(-1); q=self.shared_risk_head(z).squeeze(-1)
        if self.personalization:
            r=r+self.rul_adapter(z).squeeze(-1); q=q+self.risk_adapter(z).squeeze(-1)
        rul=torch.sigmoid(r)*(1.05*self.rul_scale)
        return rul,q


def is_private_key(k):
    return k.startswith("rul_adapter.") or k.startswith("risk_adapter.") or k in {"log_var_rul","log_var_risk"}


def split_state_dict(sd):
    private={k:v.detach().cpu().clone() for k,v in sd.items() if is_private_key(k)}
    shared={k:v.detach().cpu().clone() for k,v in sd.items() if not is_private_key(k)}
    return shared,private


def load_split_state(model,shared,private):
    sd=model.state_dict(); sd.update(shared); sd.update(private); model.load_state_dict(sd); return model


def focal_bce(logit,target,gamma=2.0,alpha=0.75):
    p=torch.sigmoid(logit); ce=F.binary_cross_entropy_with_logits(logit,target,reduction="none")
    pt=p*target+(1-p)*(1-target); at=alpha*target+(1-alpha)*(1-target)
    return (at*(1-pt).pow(gamma)*ce).mean()


def task_losses(model,rul,risk_logit,y,f,use_focal=False):
    reg=F.mse_loss(rul/model.rul_scale,y/model.rul_scale)
    if use_focal: cls=focal_bce(risk_logit,f)
    else:
        pos=max(float(f.sum().detach().cpu()),1.0); neg=max(float((1-f).sum().detach().cpu()),1.0)
        pw=torch.tensor(neg/pos,device=f.device,dtype=torch.float32)
        cls=F.binary_cross_entropy_with_logits(risk_logit,f,pos_weight=pw)
    return reg,cls


def weighted_multitask_loss(model,reg,cls,adaptive=True):
    if adaptive:
        sr=torch.clamp(model.log_var_rul,-3.,3.); sf=torch.clamp(model.log_var_risk,-3.,3.)
        return torch.exp(-sr)*reg+0.5*sr + torch.exp(-sf)*cls+0.5*sf
    return reg+0.5*cls


def shared_representation_parameters(model):
    modules=[model.sensor_encoder,model.temporal,model.gru,model.trunk]
    return [p for m in modules for p in m.parameters() if p.requires_grad]


In [6]:
# ============================================================
# Training, calibration metrics, PCGrad, and task-wise GroupDRO aggregation
# ============================================================
def make_loader(split,batch=BATCH_SIZE,shuffle=True):
    ds=TensorDataset(torch.from_numpy(split["X"]),torch.from_numpy(split["y"]),torch.from_numpy(split["f"]))
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,drop_last=False)


def _feature_ids(split,device):
    return torch.as_tensor(split["feature_ids"],dtype=torch.long,device=device)


def _grad_vector(grads,params):
    return torch.cat([(torch.zeros_like(p) if g is None else g).reshape(-1) for g,p in zip(grads,params)])


def _set_projected_grads(params,g1,g2):
    v1=_grad_vector(g1,params); v2=_grad_vector(g2,params)
    dot=torch.dot(v1,v2); n1=torch.dot(v1,v1); n2=torch.dot(v2,v2)
    cos=(dot/(torch.sqrt(n1*n2)+1e-12)).detach().cpu().item()
    if dot < 0:
        p1=v1-dot/(n2+1e-12)*v2
        p2=v2-dot/(n1+1e-12)*v1
        v=p1+p2
    else: v=v1+v2
    off=0
    for p in params:
        n=p.numel(); p.grad=v[off:off+n].view_as(p).clone(); off+=n
    return cos


def train_local_proposed(model,split,epochs=1,lr=LR,adaptive=True,use_focal=False,
                         use_pcgrad=USE_PCGRAD,freeze_shared=False,device=DEVICE):
    model=model.to(device).train()
    for name,p in model.named_parameters(): p.requires_grad=(is_private_key(name) if freeze_shared else True)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=1e-4)
    ids=_feature_ids(split,device); cosines=[]
    for _ in range(epochs):
        for X,y,f in make_loader(split):
            X,y,f=X.to(device),y.to(device),f.to(device); opt.zero_grad(set_to_none=True)
            rul,logit=model(X,ids); reg,cls=task_losses(model,rul,logit,y,f,use_focal)
            total=weighted_multitask_loss(model,reg,cls,adaptive)
            rep_params=shared_representation_parameters(model) if (use_pcgrad and not freeze_shared) else []
            if rep_params:
                sr=torch.clamp(model.log_var_rul,-3.,3.) if adaptive else torch.tensor(0.,device=device)
                sf=torch.clamp(model.log_var_risk,-3.,3.) if adaptive else torch.tensor(0.,device=device)
                lr_task=(torch.exp(-sr)*reg if adaptive else reg)
                lf_task=(torch.exp(-sf)*cls if adaptive else 0.5*cls)
                g1=torch.autograd.grad(lr_task,rep_params,retain_graph=True,allow_unused=True)
                g2=torch.autograd.grad(lf_task,rep_params,retain_graph=True,allow_unused=True)
                total.backward()
                cosines.append(_set_projected_grads(rep_params,g1,g2))
            else: total.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],5.0)
            opt.step()
    return model,{"mean_grad_cos":float(np.mean(cosines)) if cosines else np.nan,
                  "conflict_rate":float(np.mean(np.asarray(cosines)<0)) if cosines else np.nan}


def expected_calibration_error(y,p,n_bins=10):
    y=np.asarray(y);p=np.asarray(p); edges=np.linspace(0,1,n_bins+1); ece=0.
    for lo,hi in zip(edges[:-1],edges[1:]):
        m=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
        if m.any(): ece+=m.mean()*abs(y[m].mean()-p[m].mean())
    return float(ece)


@torch.no_grad()
def predict_model(model,split,device=DEVICE):
    model=model.to(device).eval(); ids=_feature_ids(split,device)
    ys=[];rs=[];fs=[];ps=[]
    for X,y,f in make_loader(split,batch=512,shuffle=False):
        rul,logit=model(X.to(device),ids)
        ys.append(y.numpy());fs.append(f.numpy());rs.append(rul.cpu().numpy());ps.append(torch.sigmoid(logit).cpu().numpy())
    return np.concatenate(ys),np.concatenate(rs),np.concatenate(fs),np.concatenate(ps)


def metrics_from_predictions(y,r,f,p,rul_scale):
    pred=(p>=.5).astype(int); eps=1e-7
    out={
        "rmse":float(np.sqrt(mean_squared_error(y,r))),"mae":float(mean_absolute_error(y,r)),
        "r2":float(r2_score(y,r)) if len(y)>1 else np.nan,
        "rul_mse_norm":float(np.mean(((y-r)/max(rul_scale,1e-8))**2)),
        "nll":float(log_loss(f,np.clip(p,eps,1-eps),labels=[0,1])),
        "brier":float(brier_score_loss(f,p)),"ece":expected_calibration_error(f,p),
    }
    if len(np.unique(f))>1:
        tn,fp,fn,tp=confusion_matrix(f,pred,labels=[0,1]).ravel()
        out.update(auprc=float(average_precision_score(f,p)),auroc=float(roc_auc_score(f,p)),
                   f1=float(f1_score(f,pred,zero_division=0)),precision=float(precision_score(f,pred,zero_division=0)),
                   recall=float(recall_score(f,pred,zero_division=0)),fnr=float(fn/max(fn+tp,1)))
    return out


def evaluate_model(model,split,device=DEVICE):
    y,r,f,p=predict_model(model,split,device)
    return metrics_from_predictions(y,r,f,p,model.rul_scale)


def aggregate_weighted(states,weights,key_selector=None):
    w=np.asarray(weights,float);w=w/max(w.sum(),1e-12);out={}
    for k in states[0]:
        wk=w if key_selector is None else np.asarray(key_selector(k),float)
        wk=wk/max(wk.sum(),1e-12)
        out[k]=sum(float(a)*s[k].float() for a,s in zip(wk,states))
    return out


def aggregate_fedavg(states,n_samples): return aggregate_weighted(states,n_samples)


def update_groupdro(q,losses,eta=DRO_ETA):
    q=np.asarray(q,float); L=np.asarray(losses,float)
    centered=np.clip(L-L.mean(),-3,3)
    z=np.log(np.clip(q,1e-12,None))+eta*centered
    z-=z.max(); q=np.exp(z); return q/q.sum()


def taskwise_aggregate(states,selected,q_rul,q_risk,taskwise=True):
    qr=np.asarray([q_rul[i] for i in selected],float); qf=np.asarray([q_risk[i] for i in selected],float)
    qr/=qr.sum();qf/=qf.sum(); qb=(qr+qf)/2; qb/=qb.sum()
    if not taskwise:
        q=(qr+qf)/2; q/=q.sum(); qr=qf=qb=q
    def sel(k):
        if k.startswith("shared_rul_head."): return qr
        if k.startswith("shared_risk_head."): return qf
        return qb
    return aggregate_weighted(states,qb,sel),qb,qr,qf


def summarize_client_metrics(metrics):
    keys=sorted({k for m in metrics for k in m})
    mean={k:float(np.nanmean([m.get(k,np.nan) for m in metrics])) for k in keys}
    mean["worst_rmse"]=float(np.nanmax([m["rmse"] for m in metrics]))
    if any("auprc" in m for m in metrics): mean["worst_auprc"]=float(np.nanmin([m.get("auprc",np.nan) for m in metrics]))
    return mean


def federated_train_proposed(clients,vocab_size,rul_scale,rounds=ROUNDS,local_epochs=LOCAL_EPOCHS,
                             adaptive=True,robust_agg=True,taskwise=True,personalization=True,
                             use_pcgrad=USE_PCGRAD,dro_eta=DRO_ETA,participation_rate=PARTICIPATION_RATE,
                             use_focal=False,verbose=True,seed=SEED):
    tic=time.perf_counter(); template=HeteroFedMTL(vocab_size,rul_scale,personalization=personalization)
    shared,_=split_state_dict(template.state_dict()); privates=[]
    for _ in clients:
        _,p=split_state_dict(HeteroFedMTL(vocab_size,rul_scale,personalization=personalization).state_dict()); privates.append(p)
    K=len(clients); ns=np.asarray([len(c["train"]["X"]) for c in clients],float)
    q_rul=np.ones(K)/K; q_risk=np.ones(K)/K; history=[]; weight_history=[]; rng=np.random.default_rng(seed)
    for rnd in range(rounds):
        msel=max(1,int(math.ceil(K*participation_rate)))
        selected=np.sort(rng.choice(K,size=msel,replace=False))
        states=[]; val_stats=[]; local_diag=[]
        for cid in selected:
            c=clients[cid]; m=HeteroFedMTL(vocab_size,rul_scale,personalization=personalization); load_split_state(m,shared,privates[cid])
            m,diag=train_local_proposed(m,c["train"],local_epochs,adaptive=adaptive,use_focal=use_focal,use_pcgrad=use_pcgrad)
            s,p=split_state_dict(m.state_dict()); states.append(s); privates[cid]=p; local_diag.append(diag)
            val_stats.append(evaluate_model(m,c["val"]))
        if robust_agg:
            # Task-wise GroupDRO: high-loss clients receive more probability mass.
            Lr=[v["rul_mse_norm"] for v in val_stats]; Lf=[v["nll"] for v in val_stats]
            qr_sel=update_groupdro(q_rul[selected],Lr,dro_eta); qf_sel=update_groupdro(q_risk[selected],Lf,dro_eta)
            q_rul[selected]=qr_sel; q_risk[selected]=qf_sel; q_rul/=q_rul.sum(); q_risk/=q_risk.sum()
            shared,qb,wr,wf=taskwise_aggregate(states,selected,q_rul,q_risk,taskwise)
        else:
            w=ns[selected]/ns[selected].sum(); shared=aggregate_weighted(states,w); qb=wr=wf=w
        per=[]
        for cid,c in enumerate(clients):
            m=HeteroFedMTL(vocab_size,rul_scale,personalization=personalization); load_split_state(m,shared,privates[cid]); per.append(evaluate_model(m,c["test"]))
        avg=summarize_client_metrics(per)
        avg["mean_grad_cos"]=float(np.nanmean([d["mean_grad_cos"] for d in local_diag]))
        avg["conflict_rate"]=float(np.nanmean([d["conflict_rate"] for d in local_diag]))
        history.append({"round":rnd+1,"mean":avg,"per_client":per,"selected":selected.tolist()})
        weight_history.append({"selected":selected.tolist(),"backbone":qb.copy(),"rul":wr.copy(),"risk":wf.copy(),"q_rul":q_rul.copy(),"q_risk":q_risk.copy()})
        if verbose: print(f"round {rnd+1:02d} | RMSE={avg['rmse']:.3f} AUPRC={avg.get('auprc',np.nan):.3f} worstRMSE={avg['worst_rmse']:.3f} conflict={avg['conflict_rate']:.2f}")
    return {"shared":shared,"privates":privates,"history":history,"weights":weight_history,
            "vocab_size":vocab_size,"rul_scale":rul_scale,"personalization":personalization,
            "elapsed_sec":time.perf_counter()-tic}


def proposed_client_model(bundle,client_id):
    m=HeteroFedMTL(bundle["vocab_size"],bundle["rul_scale"],personalization=bundle["personalization"])
    return load_split_state(m,bundle["shared"],bundle["privates"][client_id])


In [7]:
# ============================================================
# Baselines: Local, Centralized, FedAvg, FedProx, SCAFFOLD, FedPer, Ditto
# Conventional baselines use the union feature space with zero filling.
# ============================================================
class SharedMTLNet(nn.Module):
    def __init__(self,n_features,rul_scale,d_model=D_MODEL,hidden=HIDDEN):
        super().__init__();self.rul_scale=float(rul_scale)
        self.encoder=nn.Sequential(nn.Linear(n_features,d_model),nn.GELU(),nn.LayerNorm(d_model))
        self.temporal=nn.Sequential(nn.Conv1d(d_model,d_model,5,padding=2),nn.GELU(),nn.Conv1d(d_model,d_model,3,padding=1),nn.GELU())
        self.gru=nn.GRU(d_model,hidden,batch_first=True)
        self.rul_head=nn.Sequential(nn.Linear(hidden,48),nn.GELU(),nn.Linear(48,1))
        self.risk_head=nn.Sequential(nn.Linear(hidden,48),nn.GELU(),nn.Linear(48,1))
        self.log_var_rul=nn.Parameter(torch.zeros(()));self.log_var_risk=nn.Parameter(torch.zeros(()))
    def forward(self,x):
        z=self.encoder(x);z=self.temporal(z.transpose(1,2)).transpose(1,2);z,_=self.gru(z);z=z[:,-1]
        rul=torch.sigmoid(self.rul_head(z).squeeze(-1))*(1.05*self.rul_scale)
        return rul,self.risk_head(z).squeeze(-1)


def baseline_loss(model,rul,logit,y,f,mode="fixed_mtl"):
    reg=F.mse_loss(rul/model.rul_scale,y/model.rul_scale)
    pos=max(float(f.sum().detach().cpu()),1.);neg=max(float((1-f).sum().detach().cpu()),1.)
    cls=F.binary_cross_entropy_with_logits(logit,f,pos_weight=torch.tensor(neg/pos,device=f.device))
    if mode=="rul_only": return reg
    if mode=="risk_only": return cls
    if mode=="uncertainty_mtl":
        sr=torch.clamp(model.log_var_rul,-3.,3.);sf=torch.clamp(model.log_var_risk,-3.,3.)
        return torch.exp(-sr)*reg+0.5*sr+torch.exp(-sf)*cls+0.5*sf
    return reg+0.5*cls


def train_shared_local(model,split,epochs,lr=LR,global_state=None,prox_mu=0.,mode="fixed_mtl"):
    model=model.to(DEVICE).train();opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4);steps=0
    for _ in range(epochs):
        for X,y,f in make_loader(split):
            X,y,f=X.to(DEVICE),y.to(DEVICE),f.to(DEVICE);opt.zero_grad(set_to_none=True)
            rul,logit=model(X);loss=baseline_loss(model,rul,logit,y,f,mode)
            if prox_mu>0 and global_state is not None:
                prox=sum((p-global_state[n].to(DEVICE)).pow(2).sum() for n,p in model.named_parameters())
                loss=loss+0.5*prox_mu*prox
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.);opt.step();steps+=1
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()},steps


@torch.no_grad()
def evaluate_shared(model,split):
    model=model.to(DEVICE).eval();ys=[];rs=[];fs=[];ps=[]
    for X,y,f in make_loader(split,batch=512,shuffle=False):
        rul,logit=model(X.to(DEVICE));ys.append(y.numpy());fs.append(f.numpy());rs.append(rul.cpu().numpy());ps.append(torch.sigmoid(logit).cpu().numpy())
    return metrics_from_predictions(np.concatenate(ys),np.concatenate(rs),np.concatenate(fs),np.concatenate(ps),model.rul_scale)


def federated_shared_baseline(clients,rul_scale,rounds=ROUNDS,local_epochs=LOCAL_EPOCHS,prox_mu=0.,mode="fixed_mtl",verbose=False):
    tic=time.perf_counter();nfeat=clients[0]["train"]["X"].shape[-1];gm=SharedMTLNet(nfeat,rul_scale);ns=[len(c["train"]["X"]) for c in clients];hist=[]
    for rnd in range(rounds):
        g={k:v.detach().cpu().clone() for k,v in gm.state_dict().items()};states=[]
        for c in clients:
            lm=SharedMTLNet(nfeat,rul_scale);lm.load_state_dict(g);st,_=train_shared_local(lm,c["train"],local_epochs,global_state=g,prox_mu=prox_mu,mode=mode);states.append(st)
        gm.load_state_dict(aggregate_fedavg(states,ns));per=[evaluate_shared(gm,c["test"]) for c in clients];hist.append(summarize_client_metrics(per))
        if verbose: print(f"baseline {rnd+1:02d} RMSE={hist[-1]['rmse']:.3f}")
    return gm,hist,{"elapsed_sec":time.perf_counter()-tic}


def local_only_baseline(clients,rul_scale):
    per=[]
    for c in clients:
        m=SharedMTLNet(c["train"]["X"].shape[-1],rul_scale);st,_=train_shared_local(m,c["train"],max(2,LOCAL_EPOCHS*ROUNDS//2));m.load_state_dict(st);per.append(evaluate_shared(m,c["test"]))
    return summarize_client_metrics(per)


def centralized_baseline(clients,rul_scale):
    nfeat=clients[0]["train"]["X"].shape[-1]
    X=np.concatenate([c["train"]["X"] for c in clients]);y=np.concatenate([c["train"]["y"] for c in clients]);f=np.concatenate([c["train"]["f"] for c in clients])
    tr=_pack_split(X,y,f,clients[0]["train"]["features"],"centralized")
    m=SharedMTLNet(nfeat,rul_scale);st,_=train_shared_local(m,tr,max(2,LOCAL_EPOCHS*ROUNDS//2));m.load_state_dict(st)
    return summarize_client_metrics([evaluate_shared(m,c["test"]) for c in clients])


def scaffold_baseline(clients,rul_scale,rounds=ROUNDS,local_epochs=LOCAL_EPOCHS,lr=5e-3,verbose=False):
    """Self-contained SCAFFOLD baseline with SGD control variates."""
    nfeat=clients[0]["train"]["X"].shape[-1];gm=SharedMTLNet(nfeat,rul_scale);names=[n for n,p in gm.named_parameters()]
    cg={n:torch.zeros_like(p.detach().cpu()) for n,p in gm.named_parameters()}
    ci=[{n:torch.zeros_like(v) for n,v in cg.items()} for _ in clients];ns=np.asarray([len(c["train"]["X"]) for c in clients],float);hist=[]
    for rnd in range(rounds):
        g={k:v.detach().cpu().clone() for k,v in gm.state_dict().items()};states=[];new_ci=[]
        for cid,c in enumerate(clients):
            lm=SharedMTLNet(nfeat,rul_scale);lm.load_state_dict(g);lm.to(DEVICE).train();opt=torch.optim.SGD(lm.parameters(),lr=lr,momentum=0.0);K=0
            for _ in range(local_epochs):
                for X,y,f in make_loader(c["train"]):
                    X,y,f=X.to(DEVICE),y.to(DEVICE),f.to(DEVICE);opt.zero_grad(set_to_none=True);r,l=lm(X);loss=baseline_loss(lm,r,l,y,f,"fixed_mtl");loss.backward()
                    for n,p in lm.named_parameters():
                        if p.grad is not None: p.grad.add_(cg[n].to(DEVICE)-ci[cid][n].to(DEVICE))
                    torch.nn.utils.clip_grad_norm_(lm.parameters(),5.);opt.step();K+=1
            st={k:v.detach().cpu().clone() for k,v in lm.state_dict().items()};states.append(st)
            cnew={}
            for n,p in lm.named_parameters(): cnew[n]=ci[cid][n]-cg[n]+(g[n]-st[n])/(max(K,1)*lr)
            new_ci.append(cnew)
        gm.load_state_dict(aggregate_fedavg(states,ns))
        for n in names: cg[n]=cg[n]+sum(new_ci[i][n]-ci[i][n] for i in range(len(clients)))/len(clients)
        ci=new_ci;per=[evaluate_shared(gm,c["test"]) for c in clients];hist.append(summarize_client_metrics(per))
        if verbose: print("SCAFFOLD",rnd+1,hist[-1]["rmse"])
    return gm,hist


def fedper_baseline(clients,rul_scale,rounds=ROUNDS,local_epochs=LOCAL_EPOCHS):
    nfeat=clients[0]["train"]["X"].shape[-1];template=SharedMTLNet(nfeat,rul_scale)
    shared_keys=[k for k in template.state_dict() if not (k.startswith("rul_head.") or k.startswith("risk_head.") or k.startswith("log_var"))]
    g={k:v.detach().cpu().clone() for k,v in template.state_dict().items()};private=[]
    for _ in clients: private.append({k:v.detach().cpu().clone() for k,v in g.items() if k not in shared_keys})
    ns=np.asarray([len(c["train"]["X"]) for c in clients],float);hist=[]
    for _ in range(rounds):
        local=[]
        for cid,c in enumerate(clients):
            m=SharedMTLNet(nfeat,rul_scale);sd=m.state_dict();sd.update({k:g[k] for k in shared_keys});sd.update(private[cid]);m.load_state_dict(sd);st,_=train_shared_local(m,c["train"],local_epochs)
            private[cid]={k:st[k] for k in st if k not in shared_keys};local.append({k:st[k] for k in shared_keys})
        w=ns/ns.sum()
        for k in shared_keys: g[k]=sum(float(a)*s[k].float() for a,s in zip(w,local))
        per=[]
        for cid,c in enumerate(clients):
            m=SharedMTLNet(nfeat,rul_scale);sd=m.state_dict();sd.update({k:g[k] for k in shared_keys});sd.update(private[cid]);m.load_state_dict(sd);per.append(evaluate_shared(m,c["test"]))
        hist.append(summarize_client_metrics(per))
    return hist


def ditto_baseline(clients,rul_scale,rounds=ROUNDS,local_epochs=LOCAL_EPOCHS,lam=1e-2):
    nfeat=clients[0]["train"]["X"].shape[-1];global_m=SharedMTLNet(nfeat,rul_scale);pstates=[copy.deepcopy(global_m.state_dict()) for _ in clients];ns=[len(c["train"]["X"]) for c in clients];hist=[]
    for _ in range(rounds):
        g={k:v.detach().cpu().clone() for k,v in global_m.state_dict().items()};states=[]
        for c in clients:
            lm=SharedMTLNet(nfeat,rul_scale);lm.load_state_dict(g);st,_=train_shared_local(lm,c["train"],local_epochs);states.append(st)
        global_m.load_state_dict(aggregate_fedavg(states,ns));gnew={k:v.detach().cpu().clone() for k,v in global_m.state_dict().items()}
        per=[]
        for cid,c in enumerate(clients):
            pm=SharedMTLNet(nfeat,rul_scale);pm.load_state_dict(pstates[cid]);st,_=train_shared_local(pm,c["train"],local_epochs,global_state=gnew,prox_mu=lam);pstates[cid]=st;pm.load_state_dict(st);per.append(evaluate_shared(pm,c["test"]))
        hist.append(summarize_client_metrics(per))
    return hist


## Experiment 1 — Main comparison

Run the proposed method against non-federated, standard FL, control-variate FL, and personalized FL baselines. All methods use the same chronological train/validation/test protocol. Conventional baselines use union-feature zero filling; the proposed method consumes variable sensor sets directly.

In [8]:
aligned_clients=align_clients_to_union(clients,FEATURE_VOCAB)
MAIN_RESULTS={}
MAIN_ARTIFACTS={}
if RUN_MAIN_COMPARISON:
    print("Local-only");MAIN_RESULTS["Local"]=local_only_baseline(clients,RUL_SCALE)
    print("Centralized");MAIN_RESULTS["Centralized"]=centralized_baseline(aligned_clients,RUL_SCALE)
    print("FedAvg");m,h,meta=federated_shared_baseline(aligned_clients,RUL_SCALE,verbose=False);MAIN_RESULTS["FedAvg"]=h[-1];MAIN_ARTIFACTS["FedAvg"]=(m,h,meta)
    print("FedProx");m,h,meta=federated_shared_baseline(aligned_clients,RUL_SCALE,prox_mu=1e-3);MAIN_RESULTS["FedProx"]=h[-1];MAIN_ARTIFACTS["FedProx"]=(m,h,meta)
    print("SCAFFOLD");m,h=scaffold_baseline(aligned_clients,RUL_SCALE);MAIN_RESULTS["SCAFFOLD"]=h[-1]
    print("FedPer");h=fedper_baseline(aligned_clients,RUL_SCALE);MAIN_RESULTS["FedPer"]=h[-1]
    print("Ditto");h=ditto_baseline(aligned_clients,RUL_SCALE);MAIN_RESULTS["Ditto"]=h[-1]
    print("HeteroFedMTL-DRO")
    PROPOSED=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,verbose=True)
    MAIN_RESULTS["HeteroFedMTL-DRO"]=PROPOSED["history"][-1]["mean"];MAIN_ARTIFACTS["HeteroFedMTL-DRO"]=PROPOSED
    main_df=pd.DataFrame(MAIN_RESULTS).T
    show=[c for c in ["rmse","mae","r2","auprc","auroc","f1","recall","fnr","brier","ece","worst_rmse","worst_auprc"] if c in main_df.columns]
    display(main_df[show].round(4));main_df.to_csv(RESULT_DIR/"main_comparison.csv")


Local-only
Centralized
FedAvg
FedProx
SCAFFOLD
FedPer
Ditto
HeteroFedMTL-DRO
round 01 | RMSE=24.802 AUPRC=0.518 worstRMSE=29.427 conflict=nan
round 02 | RMSE=24.403 AUPRC=0.580 worstRMSE=27.315 conflict=nan
round 03 | RMSE=23.041 AUPRC=0.645 worstRMSE=26.268 conflict=nan
round 04 | RMSE=22.829 AUPRC=0.692 worstRMSE=26.444 conflict=nan
round 05 | RMSE=20.671 AUPRC=0.664 worstRMSE=24.766 conflict=nan
round 06 | RMSE=20.668 AUPRC=0.686 worstRMSE=24.784 conflict=nan
round 07 | RMSE=20.581 AUPRC=0.693 worstRMSE=24.108 conflict=nan
round 08 | RMSE=20.716 AUPRC=0.687 worstRMSE=24.802 conflict=nan
round 09 | RMSE=20.256 AUPRC=0.676 worstRMSE=24.499 conflict=nan
round 10 | RMSE=20.995 AUPRC=0.697 worstRMSE=25.772 conflict=nan
round 11 | RMSE=21.367 AUPRC=0.676 worstRMSE=26.516 conflict=nan
round 12 | RMSE=20.740 AUPRC=0.701 worstRMSE=25.566 conflict=nan
round 13 | RMSE=22.113 AUPRC=0.688 worstRMSE=24.880 conflict=nan
round 14 | RMSE=21.863 AUPRC=0.681 worstRMSE=25.357 conflict=nan
round 15 | RM

,rmse,mae,r2,auprc,auroc,f1,recall,fnr,brier,ece,worst_rmse,worst_auprc
Local,19.0361,12.5438,0.5538,0.7369,0.9700,0.6732,0.6717,0.3283,0.0182,0.0178,21.6303,0.5046
Centralized,19.3825,12.5882,0.5421,0.7997,0.9848,0.7352,0.7647,0.2353,0.0148,0.0144,21.4878,0.6708
FedAvg,24.4909,15.2399,0.2012,0.7354,0.9845,0.5443,0.7842,0.2158,0.0398,0.0430,33.0999,0.6347
FedProx,23.3637,14.8703,0.2863,0.7551,0.9814,0.5986,0.7305,0.2695,0.0252,0.0265,30.2788,0.6289
SCAFFOLD,18.2909,13.7066,0.5920,0.7682,0.9879,0.6840,0.8023,0.1977,0.0162,0.0171,19.8473,0.6609
FedPer,21.1771,14.4921,0.4514,0.6912,0.9769,0.6473,0.7533,0.2467,0.0209,0.0207,22.5147,0.6044
Ditto,19.6616,12.7887,0.5287,0.7897,0.9825,0.7352,0.7266,0.2734,0.0140,0.0130,21.3936,0.6388
HeteroFedMTL-DRO,20.8208,13.5535,0.4727,0.6829,0.9709,0.5861,0.5583,0.4417,0.0209,0.0208,23.3557,0.4862


## Experiment 2 — Is multi-task learning actually useful?

This isolates the MTL claim: RUL-only, risk-only, fixed-weight MTL, uncertainty-weighted MTL, and the full proposed method. A top-tier paper should show that joint learning improves the task(s) rather than merely adding a second output head.

In [9]:
# if RUN_MTL_STUDY:
#     rows={}
#     for mode in ["rul_only","risk_only","fixed_mtl","uncertainty_mtl"]:
#         print("MTL baseline:",mode)
#         _,h,_=federated_shared_baseline(aligned_clients,RUL_SCALE,mode=mode)
#         rows[mode]=h[-1]
#     b=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,verbose=False)
#     rows["HeteroFedMTL-DRO"]=b["history"][-1]["mean"]
#     mtl_df=pd.DataFrame(rows).T;display(mtl_df[[c for c in ["rmse","mae","auprc","f1","worst_rmse"] if c in mtl_df]].round(4));mtl_df.to_csv(RESULT_DIR/"mtl_study.csv")


## Experiment 3 — Mechanism ablations

Ablate the optimization and personalization mechanisms individually. Feature-set handling is isolated separately in the sensor-missingness stress test rather than conflated with the optimization ablation.

In [10]:
# if RUN_ABLATIONS:
#     variants={
#         "Full":dict(adaptive=True,robust_agg=True,taskwise=True,personalization=True,use_pcgrad=True),
#         "-PCGrad":dict(adaptive=True,robust_agg=True,taskwise=True,personalization=True,use_pcgrad=False),
#         "-GroupDRO":dict(adaptive=True,robust_agg=False,taskwise=False,personalization=True,use_pcgrad=True),
#         "-TaskwiseDRO":dict(adaptive=True,robust_agg=True,taskwise=False,personalization=True,use_pcgrad=True),
#         "-Personalization":dict(adaptive=True,robust_agg=True,taskwise=True,personalization=False,use_pcgrad=True),
#         "-AdaptiveLoss":dict(adaptive=False,robust_agg=True,taskwise=True,personalization=True,use_pcgrad=True),
#     }
#     rows={}
#     for name,kw in variants.items():
#         print("Ablation",name);b=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,verbose=False,**kw);rows[name]=b["history"][-1]["mean"]
#     ablation_df=pd.DataFrame(rows).T;display(ablation_df[[c for c in ["rmse","auprc","worst_rmse","worst_auprc","conflict_rate"] if c in ablation_df]].round(4));ablation_df.to_csv(RESULT_DIR/"ablations.csv")


## Experiment 4 — Feature heterogeneity stress test

Sweep available sensor fractions and compare random/structured missingness. The key comparison is against standard union zero-filling and, when possible, feature-intersection FedAvg.

In [11]:
# if RUN_FEATURE_STRESS:
#     rows=[];stress_rounds=25 if PAPER_MODE else 2
#     for mode in ["random","structured"]:
#         for frac in [1.0,0.8,0.6,0.4,0.2]:
#             print("feature stress",mode,frac)
#             cs,vocab,rs=finalize_clients(copy.deepcopy(RAW_CLIENTS),min_fraction=frac,feature_mode=("none" if frac==1 else mode),seed=SEED)
#             b=federated_train_proposed(cs,len(vocab),rs,rounds=stress_rounds,verbose=False);m=b["history"][-1]["mean"]
#             rows.append({"method":"HeteroFedMTL-DRO","mode":mode,"feature_fraction":frac,**m})
#             al=align_clients_to_union(cs,vocab);_,h,_=federated_shared_baseline(al,rs,rounds=stress_rounds);rows.append({"method":"ZeroFill-FedAvg","mode":mode,"feature_fraction":frac,**h[-1]})
#             try:
#                 inter,_=align_clients_to_intersection(cs);_,hi,_=federated_shared_baseline(inter,rs,rounds=stress_rounds);rows.append({"method":"Intersection-FedAvg","mode":mode,"feature_fraction":frac,**hi[-1]})
#             except ValueError: pass
#     feature_df=pd.DataFrame(rows);display(feature_df[["method","mode","feature_fraction","rmse","auprc","worst_rmse"]].round(4));feature_df.to_csv(RESULT_DIR/"feature_stress.csv",index=False)


## Experiment 5 — Distribution heterogeneity stress test

Increase client-specific RUL distribution shift while keeping each client training-set size fixed through weighted resampling. This measures whether robustness comes from the method rather than from an easy partition.

In [12]:
# if RUN_NONIID_STRESS:
#     rows=[];stress_rounds=25 if PAPER_MODE else 2
#     for sev in [0.0,0.5,1.0,2.0,4.0]:
#         cs=induce_rul_distribution_shift(clients,sev,SEED)
#         b=federated_train_proposed(cs,len(FEATURE_VOCAB),RUL_SCALE,rounds=stress_rounds,verbose=False);rows.append({"method":"HeteroFedMTL-DRO","severity":sev,**b["history"][-1]["mean"]})
#         al=align_clients_to_union(cs,FEATURE_VOCAB);_,h,_=federated_shared_baseline(al,RUL_SCALE,rounds=stress_rounds);rows.append({"method":"FedAvg","severity":sev,**h[-1]})
#     noniid_df=pd.DataFrame(rows);display(noniid_df[["method","severity","rmse","auprc","worst_rmse"]].round(4));noniid_df.to_csv(RESULT_DIR/"noniid_stress.csv",index=False)


## Experiment 6 — Partial client participation

Test robustness when only a subset of clients participates each round. This is a core FL stress test and should be reported with identical random seeds across participation rates.

In [13]:
# if RUN_PARTICIPATION_STRESS:
#     rows=[];stress_rounds=30 if PAPER_MODE else 2
#     for rate in [1.0,0.75,0.5,0.25,0.10]:
#         b=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,rounds=stress_rounds,participation_rate=rate,verbose=False,seed=SEED)
#         rows.append({"participation":rate,**b["history"][-1]["mean"]})
#     participation_df=pd.DataFrame(rows);display(participation_df[["participation","rmse","auprc","worst_rmse"]].round(4));participation_df.to_csv(RESULT_DIR/"participation_stress.csv",index=False)


## Experiment 7 — Leave-one-client-out zero-shot and few-shot generalization

Every client is held out once. The sensor vocabulary is built **only from training clients**; previously unseen sensor names map to an UNK sensor embedding. Evaluate zero-shot and 1/5/10/20/50% labeled adaptation.

In [14]:
# def _slice_prefix(split,fraction):
#     n=max(1,int(len(split["X"])*fraction));out={k:(v[:n] if isinstance(v,np.ndarray) and len(v)==len(split["X"]) else copy.deepcopy(v)) for k,v in split.items()};return out


# def leave_one_client_out(all_clients,holdout,fractions=(0,.01,.05,.10,.20,.50),rounds=None):
#     h=holdout%len(all_clients);train_cs=copy.deepcopy([c for i,c in enumerate(all_clients) if i!=h]);held=copy.deepcopy(all_clients[h])
#     vocab=build_feature_vocab(train_cs);assign_feature_ids(train_cs,vocab);assign_feature_ids([held],vocab,allow_unknown=True)
#     rs=max(max(float(np.max(c["train"]["y"])),1.) for c in train_cs);vsize=len(vocab)+1
#     b=federated_train_proposed(train_cs,vsize,rs,rounds=(rounds or ROUNDS),verbose=False)
#     rows=[]
#     for frac in fractions:
#         m=HeteroFedMTL(vsize,rs,personalization=True);_,fresh=split_state_dict(m.state_dict());load_split_state(m,b["shared"],fresh)
#         if frac>0:
#             fs=_slice_prefix(held["train"],frac);m,_=train_local_proposed(m,fs,epochs=max(3,LOCAL_EPOCHS),freeze_shared=True,use_pcgrad=False)
#         rows.append({"heldout":held["name"],"adapt_fraction":frac,**evaluate_model(m,held["test"])})
#     return rows

# if RUN_UNSEEN_CLIENT:
#     rows=[]
#     for h in range(len(clients)):
#         print("LOCO holdout",clients[h]["name"]);rows.extend(leave_one_client_out(clients,h,rounds=(30 if PAPER_MODE else 2)))
#     unseen_df=pd.DataFrame(rows);display(unseen_df.groupby("adapt_fraction")[["rmse","auprc","worst_rmse"] if "worst_rmse" in unseen_df else ["rmse","auprc"]].mean().round(4));unseen_df.to_csv(RESULT_DIR/"unseen_client.csv",index=False)


## Experiment 8 — Calibration and reliability

Failure-risk probabilities should be calibrated, not just discriminative. Report NLL, Brier score, ECE, and a reliability diagram.

In [15]:
# def reliability_curve_data(y,p,n_bins=10):
#     edges=np.linspace(0,1,n_bins+1);rows=[]
#     for lo,hi in zip(edges[:-1],edges[1:]):
#         m=(p>=lo)&(p<(hi if hi<1 else hi+1e-9))
#         if m.any(): rows.append({"confidence":float(p[m].mean()),"frequency":float(y[m].mean()),"n":int(m.sum())})
#     return pd.DataFrame(rows)

# if RUN_CALIBRATION:
#     if "PROPOSED" not in globals(): PROPOSED=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,verbose=False)
#     rows=[];all_y=[];all_p=[]
#     for cid,c in enumerate(clients):
#         m=proposed_client_model(PROPOSED,cid);y,r,f,p=predict_model(m,c["test"]);rows.append({"client":c["name"],**metrics_from_predictions(y,r,f,p,RUL_SCALE)});all_y.append(f);all_p.append(p)
#     calibration_df=pd.DataFrame(rows);display(calibration_df[["client","nll","brier","ece","auprc"]].round(4));calibration_df.to_csv(RESULT_DIR/"calibration.csv",index=False)
#     rc=reliability_curve_data(np.concatenate(all_y),np.concatenate(all_p));plt.figure(figsize=(5,5));plt.plot([0,1],[0,1],"--");plt.plot(rc.confidence,rc.frequency,marker="o");plt.xlabel("Predicted probability");plt.ylabel("Observed frequency");plt.title("Reliability diagram");plt.show()


## Experiment 9 — Efficiency and communication cost

Report trainable parameters, communicated shared bytes per client/round, wall-clock time, and convergence rounds. The code below avoids third-party FLOP packages; add hardware details in the paper.

In [16]:
# def parameter_count(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
# def state_bytes(sd): return sum(v.numel()*v.element_size() for v in sd.values())

# def first_round_to_fraction(history,key="rmse",fraction=1.05):
#     vals=np.asarray([h["mean"][key] if "mean" in h else h[key] for h in history],float);target=vals.min()*fraction
#     idx=np.where(vals<=target)[0];return int(idx[0]+1) if len(idx) else len(vals)

# if RUN_EFFICIENCY:
#     if "PROPOSED" not in globals(): PROPOSED=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,verbose=False)
#     pm=HeteroFedMTL(len(FEATURE_VOCAB),RUL_SCALE);ps,_=split_state_dict(pm.state_dict())
#     sm=SharedMTLNet(len(FEATURE_VOCAB),RUL_SCALE)
#     rows=[
#         {"method":"HeteroFedMTL-DRO","params":parameter_count(pm),"shared_MB":state_bytes(ps)/1e6,"MB_round_full_participation":2*len(clients)*state_bytes(ps)/1e6,"elapsed_sec":PROPOSED["elapsed_sec"],"round_to_5pct_best":first_round_to_fraction(PROPOSED["history"])},
#         {"method":"FedAvg","params":parameter_count(sm),"shared_MB":state_bytes(sm.state_dict())/1e6,"MB_round_full_participation":2*len(clients)*state_bytes(sm.state_dict())/1e6,"elapsed_sec":MAIN_ARTIFACTS.get("FedAvg",(None,None,{"elapsed_sec":np.nan}))[2]["elapsed_sec"] if "MAIN_ARTIFACTS" in globals() else np.nan}
#     ]
#     efficiency_df=pd.DataFrame(rows);display(efficiency_df.round(3));efficiency_df.to_csv(RESULT_DIR/"efficiency.csv",index=False)


## Experiment 10 — Gradient conflict analysis

The proposed trainer logs cosine similarity between RUL and risk gradients on the shared representation and the fraction of batches with negative cosine. This directly tests whether PCGrad addresses negative transfer.

In [17]:
# if "PROPOSED" in globals():
#     grad_df=pd.DataFrame([{"round":h["round"],"mean_grad_cos":h["mean"]["mean_grad_cos"],"conflict_rate":h["mean"]["conflict_rate"]} for h in PROPOSED["history"]])
#     display(grad_df.round(4));plt.figure(figsize=(8,4));plt.plot(grad_df["round"],grad_df["conflict_rate"]);plt.xlabel("Round");plt.ylabel("Negative-gradient fraction");plt.title("Task gradient conflict");plt.show()


## Experiment 11 — Multi-seed statistics and paired significance

For the final table run the key methods with the same five seeds. Report mean ± standard deviation, 95% CI, and a paired Wilcoxon test for the main comparison.

In [18]:
# def ci95(x):
#     x=np.asarray(x,float);return 1.96*np.nanstd(x,ddof=1)/math.sqrt(max(np.sum(np.isfinite(x)),1)) if len(x)>1 else np.nan

# if RUN_MULTI_SEED:
#     methods=["FedAvg","FedProx","Ditto","HeteroFedMTL-DRO"];rows=[]
#     for s in range(N_SEEDS):
#         seed=100+s;seed_everything(seed)
#         _,h,_=federated_shared_baseline(aligned_clients,RUL_SCALE,mode="fixed_mtl");rows.append({"seed":seed,"method":"FedAvg",**h[-1]})
#         _,h,_=federated_shared_baseline(aligned_clients,RUL_SCALE,prox_mu=1e-3);rows.append({"seed":seed,"method":"FedProx",**h[-1]})
#         h=ditto_baseline(aligned_clients,RUL_SCALE);rows.append({"seed":seed,"method":"Ditto",**h[-1]})
#         b=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,verbose=False,seed=seed);rows.append({"seed":seed,"method":"HeteroFedMTL-DRO",**b["history"][-1]["mean"]})
#     seed_df=pd.DataFrame(rows);summary=[]
#     for m,g in seed_df.groupby("method"):
#         summary.append({"method":m,"rmse_mean":g.rmse.mean(),"rmse_std":g.rmse.std(),"rmse_ci95":ci95(g.rmse),"auprc_mean":g.auprc.mean(),"auprc_std":g.auprc.std(),"auprc_ci95":ci95(g.auprc)})
#     stats_df=pd.DataFrame(summary);display(stats_df.round(4));seed_df.to_csv(RESULT_DIR/"multiseed_raw.csv",index=False);stats_df.to_csv(RESULT_DIR/"multiseed_summary.csv",index=False)
#     prop=seed_df[seed_df.method=="HeteroFedMTL-DRO"].sort_values("seed").rmse.values
#     for m in ["FedAvg","FedProx","Ditto"]:
#         x=seed_df[seed_df.method==m].sort_values("seed").rmse.values
#         if len(x)==len(prop) and len(x)>=5: print("Wilcoxon RMSE, proposed vs",m, wilcoxon(prop,x,alternative="less"))


## Experiment 12 — Hyperparameter sensitivity

The main sensitivity parameter introduced by the method is the GroupDRO exponentiated-gradient step size. Also report local epochs and participation sensitivity if space permits.

In [19]:
# if RUN_SENSITIVITY:
#     rows=[];sens_rounds=25 if PAPER_MODE else 2
#     for eta in [0.0,0.1,0.2,0.35,0.5,0.8]:
#         b=federated_train_proposed(clients,len(FEATURE_VOCAB),RUL_SCALE,rounds=sens_rounds,dro_eta=eta,verbose=False);rows.append({"DRO_ETA":eta,**b["history"][-1]["mean"]})
#     sensitivity_df=pd.DataFrame(rows);display(sensitivity_df[["DRO_ETA","rmse","auprc","worst_rmse"]].round(4));sensitivity_df.to_csv(RESULT_DIR/"sensitivity.csv",index=False)


## Final paper experiment checklist

For a serious main-track submission, do **not** report only the best run. The final paper should include:

1. Main comparison: Local, Centralized, FedAvg, FedProx, SCAFFOLD, FedPer, Ditto, and HeteroFedMTL-DRO.
2. Single-task vs fixed-MTL vs uncertainty-MTL vs proposed.
3. Mechanism ablations: PCGrad, GroupDRO, task-wise DRO, personalization, adaptive loss.
4. Random and structured sensor-missingness sweeps (100/80/60/40/20%).
5. Increasing non-IID RUL distribution shift.
6. Partial participation (100/75/50/25/10%).
7. Leave-one-client-out zero-shot and 1/5/10/20/50% adaptation.
8. Calibration (NLL, Brier, ECE, reliability plot).
9. Efficiency (parameters, communicated MB/round, wall time, rounds-to-convergence).
10. Gradient-conflict diagnostics.
11. Five matched seeds, mean±std, 95% CI, paired significance test.
12. GroupDRO sensitivity and any dataset-specific horizon sensitivity.

Run the same protocol independently on C-MAPSS, N-CMAPSS, IMS, and XJTU-SY. Treat datasets as **separate testbeds**, not as four clients in one federation. Keep all final raw result CSVs produced in `RESULT_DIR` for reproducibility.
